# Model Machine Learning: Random Forest Regressor
### Prediksi Harga Rumah (Kaggle Competition) - Mencegah Data Leakage & Imputasi Median

Notebook ini merupakan versi Jupyter Notebook yang rapi dan terstruktur dari script `RandomForestRegressor.py`.

---

### Alur Kerja & Langkah-Langkah:
1. **Load Data**: Membaca dataset latih (`train.csv`) dan dataset uji (`test.csv`).
2. **Pemilihan Fitur**: Memilih 10 fitur utama yang berkorelasi tinggi dengan harga rumah.
3. **Pemisahan Data Terlebih Dahulu (`train_test_split`)**: Menghindari kebocoran data (*Data Leakage*) sebelum melakukan pembersihan data.
4. **Imputasi Median Secara Terpisah**: Menghitung median hanya dari data latih (`train_X`) untuk mengisi nilai yang hilang pada `train_X`, `val_X`, `test_X`, dan `X`.
5. **Verifikasi Missing Values**: Memastikan tidak ada lagi nilai kosong setelah imputasi.
6. **Evaluasi Model Random Forest**: Melatih `RandomForestRegressor` dan mengukur performa MAE pada data validasi.
7. **Fit Model Akhir pada Seluruh Data**: Melatih ulang model pada keseluruhan data bersih (`X_clean`, `y`) untuk memaksimalkan akurasi.
8. **Prediksi Data Test & Pembuatan File Submission**: Menghasilkan prediksi harga pada data test dan menyiapkan output submission.


## 1. Import Library
Mengimpor pustaka yang dibutuhkan:
- `pandas` dan `numpy` untuk pengolahan dataset numerik.
- `RandomForestRegressor` dari `sklearn.ensemble` untuk model Random Forest.
- `mean_absolute_error` dari `sklearn.metrics` untuk metrik evaluasi.
- `train_test_split` dari `sklearn.model_selection` untuk membagi data.


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


## 2. Load Data & Pemilihan Fitur
Memuat `train.csv` dan `test.csv`.
Fitur target adalah `SalePrice`. Fitur independen yang digunakan adalah 10 fitur utama.


In [ ]:
# 1. Load Data
model_file_path = "train.csv" if os.path.exists("train.csv") else r"D:/file baru/mendata/machine learning competition/train.csv"
model_file_path_test = "test.csv" if os.path.exists("test.csv") else r"D:/file baru/mendata/machine learning competition/test.csv"

model_data = pd.read_csv(model_file_path)
model_data_test = pd.read_csv(model_file_path_test)

y = model_data.SalePrice
model_features = [
    'OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 
    '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'FullBath', 
    'TotRmsAbvGrd', 'LotArea'
]

X = model_data[model_features]
test_X = model_data_test[model_features]

print(f"Ukuran X: {X.shape}")
print(f"Ukuran test_X: {test_X.shape}")


## 3. Split Data Terlebih Dahulu (Mencegah Data Leakage)
> **Penting (Data Leakage):**
> Jika kita melakukan imputasi nilai kosong menggunakan median dari seluruh dataset sebelum dibagi, maka informasi dari data validasi atau data uji akan bocor ke dalam data latih.
> Oleh karena itu, langkah terbaik adalah membagi data (`train_test_split`) terlebih dahulu.


In [ ]:
# 2. SPLIT DATA TERLEBIH DAHULU (Mencegah Data Leakage)
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

print(f"Bentuk train_X: {train_X.shape}")
print(f"Bentuk val_X  : {val_X.shape}")


## 4. Imputasi Median Secara Terpisah
Menghitung median hanya berdasarkan data `train_X`.
Kemudian nilai median tersebut digunakan untuk mengisi kekosongan pada `train_X`, `val_X`, serta `test_X` dan `X`.


In [ ]:
# 3. IMPUTASI MEDIAN SECARA TERPISAH
# Gunakan median dari train_X saja untuk mengisi train_X, val_X, dan test_X
train_median = train_X.median()

print("=== NILAI MEDIAN DARI DATA TRAIN_X ===")
print(train_median)

train_X = train_X.fillna(train_median)
val_X = val_X.fillna(train_median)

# Imputasi data tes menggunakan median dari data train
test_X_clean = test_X.fillna(train_median)
X_clean = X.fillna(train_median)


## 5. Pemeriksaan Persentase Missing Value
Memverifikasi bahwa seluruh data pada `X_clean` sudah bebas dari nilai kosong (`null`).


In [ ]:
total_kolom_Xclean = X_clean.isnull().sum()
total_seluruh_Xclean = np.prod(X_clean.shape)
persentase_kehilangan = (total_kolom_Xclean / total_seluruh_Xclean) * 100

print("===== PERSENTASE VALUE HILANG =====")
print(persentase_kehilangan)


## 6. Evaluasi Model Random Forest (Tanpa Data Leakage)
Melatih model `RandomForestRegressor` dengan 100 pohon keputusan (`n_estimators=100`) pada `train_X` dan mengevaluasi MAE pada `val_X`.


In [ ]:
# 4. Evaluasi Model Random Forest
# Catatan: n_estimators biasanya bagus di angka 100-300, tidak perlu dicari dari 2..100
model_rf = RandomForestRegressor(n_estimators=100, random_state=1)
model_rf.fit(train_X, train_y)
val_predicts = model_rf.predict(val_X)

mae = mean_absolute_error(val_y, val_predicts)
print(f"MAE Validasi Random Forest (Tanpa Leakage): {mae:.2f}")


### Visualisasi Feature Importance dari Model Random Forest
Melihat fitur mana yang paling berpengaruh dalam prediksi harga rumah menurut model Random Forest.


In [ ]:
feature_importances = pd.Series(model_rf.feature_importances_, index=model_features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances.values, y=feature_importances.index, palette="viridis")
plt.title("Feature Importance (Tingkat Kepentingan Fitur) - Random Forest")
plt.xlabel("Skor Kepentingan")
plt.show()


## 7. Fit Model Akhir pada Seluruh Data (`X_clean`, `y`) & Prediksi Test
Setelah performa model terbukti baik pada data validasi, kita melatih ulang model pada keseluruhan data (`X_clean`, `y`) untuk memanfaatkan seluruh data yang tersedia sebelum memprediksi data uji (`test_X_clean`).


In [ ]:
# 5. Fit Model Akhir pada Seluruh Data (X_clean, y)
final_model = RandomForestRegressor(n_estimators=100, random_state=1)
final_model.fit(X_clean, y)
test_predict = final_model.predict(test_X_clean)

# Output Submit
output = pd.DataFrame({
    'Id': model_data_test['Id'],
    'SalePrice': test_predict
})

print("===== CONTOH OUTPUT PREDIKSI SUBMISSION =====")
display(output.head(10))

# Simpan ke file CSV
output.to_csv('submission_rf.csv', index=False)
print("\nFile 'submission_rf.csv' berhasil disimpan!")


### Visualisasi Hasil Prediksi Akhir Random Forest
Menampilkan perbandingan distribusi prediksi dengan statistik ringkas.


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(output['SalePrice'], kde=True, color='teal')
plt.title("Distribusi Harga Rumah Hasil Prediksi (Random Forest)")
plt.xlabel("SalePrice ($)")
plt.ylabel("Frekuensi")
plt.show()

print(f"Rata-rata Prediksi : ${output['SalePrice'].mean():,.2f}")
print(f"Median Prediksi    : ${output['SalePrice'].median():,.2f}")
print(f"Prediksi Terendah  : ${output['SalePrice'].min():,.2f}")
print(f"Prediksi Tertinggi : ${output['SalePrice'].max():,.2f}")
